<a href="https://colab.research.google.com/github/fezeusabrina/Telematics-driving-behavior./blob/main/Telematics_driving_behavior.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# Si vous avez mis le fichier à la racine de votre Drive :
chemin_fichier = '/content/drive/MyDrive/v2.csv'

print("Chargement des données en cours...")
# On garde l'astuce des 100 000 lignes pour coder vite ce soir !
df = pd.read_csv(chemin_fichier, nrows=100000)
print("Chargement terminé !\n")

print("--- INFOS SUR LES COLONNES ---")
print(df.info())

Chargement des données en cours...
Chargement terminé !

--- INFOS SUR LES COLONNES ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 17 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   tripID     100000 non-null  int64  
 1   deviceID   100000 non-null  float64
 2   timeStamp  100000 non-null  object 
 3   accData    100000 non-null  object 
 4   gps_speed  100000 non-null  float64
 5   battery    100000 non-null  float64
 6   cTemp      100000 non-null  float64
 7   dtc        100000 non-null  float64
 8   eLoad      100000 non-null  float64
 9   iat        100000 non-null  float64
 10  imap       100000 non-null  float64
 11  kpl        100000 non-null  float64
 12  maf        100000 non-null  float64
 13  rpm        100000 non-null  float64
 14  speed      100000 non-null  float64
 15  tAdv       100000 non-null  float64
 16  tPos       100000 non-null  float64
dtypes: float64(14), in

In [ ]:
import numpy as np

# On sélectionne les capteurs intéressants
colonnes_utiles = ['speed', 'rpm', 'eLoad', 'tPos', 'cTemp', 'battery']
df_clean = df[colonnes_utiles].copy()

# On supprime les lignes vides au cas où
df_clean = df_clean.dropna()

# Notre petite règle pour étiqueter les données
def label_driving_style(row):
    if row['speed'] > 110 and row['rpm'] > 3200:
        return 'Agressif'
    elif row['rpm'] < 1800:
        return 'Eco'
    else:
        return 'Normal'

print("Calcul des styles de conduite en cours...")
df_clean['Target'] = df_clean.apply(label_driving_style, axis=1)

print("Terminé ! Voici la répartition :")
print(df_clean['Target'].value_counts())

Calcul des styles de conduite en cours...
Terminé ! Voici la répartition :
Target
Eco         75619
Normal      24360
Agressif       21
Name: count, dtype: int64


## Cellule 4 : Préparation pour le Machine Learning

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# X : Nos capteurs (on enlève la colonne 'Target' qu'on veut deviner)
X = df_clean.drop('Target', axis=1)
# y : Notre cible
y = df_clean['Target']

# Découpage 80% Entraînement / 20% Test
# Le fameux stratify=y permet de garder la même proportion de chaque classe !
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalisation : on met tout à la même échelle
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Données prêtes et normalisées ! Prêt pour l'entraînement.")

✅ Données prêtes et normalisées ! Prêt pour l'entraînement.


## Cellule 5 : Entraînement du Random Forest (Le test ultime)texte en gras

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("🤖 Entraînement du Random Forest en cours...")
# On crée le modèle
modele = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
# On l'entraîne sur nos données
modele.fit(X_train_scaled, y_train)

# On lui fait passer l'examen sur les 20% de données cachées
print("🔍 Génération des prédictions...")
y_pred = modele.predict(X_test_scaled)

# Les résultats
print("\n--- 🏁 RÉSULTATS DU MODÈLE ---")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))

print("\nRapport de Précision :")
print(classification_report(y_test, y_pred))

🤖 Entraînement du Random Forest en cours...
🔍 Génération des prédictions...

--- 🏁 RÉSULTATS DU MODÈLE ---

Matrice de confusion :
[[    4     0     0]
 [    0 15124     0]
 [    0     0  4872]]

Rapport de Précision :
              precision    recall  f1-score   support

    Agressif       1.00      1.00      1.00         4
         Eco       1.00      1.00      1.00     15124
      Normal       1.00      1.00      1.00      4872

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



## Cellule 4 (Version "Anti-Triche")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# X : On supprime la Target, MAIS AUSSI 'speed' et 'rpm' pour empêcher l'IA de tricher !
colonnes_a_supprimer = ['Target', 'speed', 'rpm']
X = df_clean.drop(colonnes_a_supprimer, axis=1)

# y : Notre cible reste la même
y = df_clean['Target']

# On affiche ce qui reste pour l'IA
print("L'IA va s'entraîner uniquement sur ces colonnes :", list(X.columns))

# Découpage 80% / 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Données sans fuite prêtes pour le vrai test !")

L'IA va s'entraîner uniquement sur ces colonnes : ['eLoad', 'tPos', 'cTemp', 'battery']
✅ Données sans fuite prêtes pour le vrai test !


## Cellule 5 : Le vrai Entraînement

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("🤖 Entraînement du modèle (sans triche) en cours...")
modele = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
modele.fit(X_train_scaled, y_train)

print("🔍 Génération des prédictions...")
y_pred = modele.predict(X_test_scaled)

print("\n--- 🏁 LES VRAIS RÉSULTATS DU MODÈLE ---")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))

print("\nRapport de Précision :")
print(classification_report(y_test, y_pred))

🤖 Entraînement du modèle (sans triche) en cours...
🔍 Génération des prédictions...

--- 🏁 LES VRAIS RÉSULTATS DU MODÈLE ---

Matrice de confusion :
[[    0     2     2]
 [    2 13966  1156]
 [    1  3111  1760]]

Rapport de Précision :
              precision    recall  f1-score   support

    Agressif       0.00      0.00      0.00         4
         Eco       0.82      0.92      0.87     15124
      Normal       0.60      0.36      0.45      4872

    accuracy                           0.79     20000
   macro avg       0.47      0.43      0.44     20000
weighted avg       0.77      0.79      0.77     20000



## Cellule 5 (Version "Punition")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("🤖 Entraînement du modèle (avec pénalité sur les classes rares)...")

# --- LA MAGIE EST ICI : class_weight='balanced' ---
modele_force = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
# --------------------------------------------------

modele_force.fit(X_train_scaled, y_train)

print("🔍 Génération des prédictions...")
y_pred_force = modele_force.predict(X_test_scaled)

print("\n--- 🏁 RÉSULTATS AVEC CLASS_WEIGHT ---")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_force))

print("\nRapport de Précision :")
print(classification_report(y_test, y_pred_force))

🤖 Entraînement du modèle (avec pénalité sur les classes rares)...
🔍 Génération des prédictions...

--- 🏁 RÉSULTATS AVEC CLASS_WEIGHT ---

Matrice de confusion :
[[    2     1     1]
 [   10 10276  4838]
 [   10   990  3872]]

Rapport de Précision :
              precision    recall  f1-score   support

    Agressif       0.09      0.50      0.15         4
         Eco       0.91      0.68      0.78     15124
      Normal       0.44      0.79      0.57      4872

    accuracy                           0.71     20000
   macro avg       0.48      0.66      0.50     20000
weighted avg       0.80      0.71      0.73     20000

